# S3.12 — Date/Time Functions
**Date completed:** October 2026  
**Status:** Completed  
**Key concept:** Always convert strings to DateType before date operations

In [0]:
# ============================================================
# S3.12 — Cell 2
# Date and Time Functions: Silver-Layer Patterns
#
# Goal:
# 1. Convert STRING dates to DateType.
# 2. Convert STRING timestamps to TimestampType.
# 3. Safely handle more than one incoming date format.
# 4. Convert Unix epoch seconds to timestamp.
# 5. Extract date and timestamp components.
# 6. Perform date arithmetic.
# 7. Apply date functions to samples.tpch.orders.
# 8. Understand timezone and reproducibility considerations.
#
# Dataset:
# samples.tpch.orders
#
# Environment:
# Databricks Serverless
#
# Important:
# Date parsing must follow the source-system contract.
# Do not assume every incoming date uses the same format.
# ============================================================

import pyspark.sql.functions as F

print("=== S3.12: DATE AND TIME FUNCTIONS ===")
print()

# ------------------------------------------------------------
# Step 1 — Create Controlled Bronze-Like Date Data
#
# Bronze systems commonly deliver:
#
# - dates as STRING
# - timestamps as STRING
# - Unix epoch values as numbers
# - multiple date formats
# - NULL values
#
# We intentionally include all of these cases.
# ------------------------------------------------------------

bronze_dates = [

    (1001, "2026-01-15", "2026-01-15 08:30:00", 1768465800),

    (1002, "2026-02-28", "2026-02-28 14:45:22", 1772289922),

    (1003, "2026-03-31", "2026-03-31 23:59:59", 1775001599),

    # Different date format
    (1004, "15/04/2026", "2026-04-15 06:00:00", 1776232800),

    # NULL date/time values
    (1005, None, None, None)]

df_bronze = spark.createDataFrame(
    bronze_dates,
    ["order_id", "order_date_str", "order_ts_str", "unix_ts"])

print("=== STEP 1: BRONZE DATE DATA ===")

df_bronze.printSchema()

df_bronze.show(truncate=False)

source_count = df_bronze.count()

print(f"\nSource row count: {source_count}")

print()


In [0]:
# ------------------------------------------------------------
# Step 2 — Inspect Session Timezone
#
# Date values do not contain time-of-day information.
#
# Timestamp display and conversion can be affected by the
# Spark session timezone.
#
# This is particularly important when processing:
#
# - Unix timestamps
# - UTC source timestamps
# - systems operating across several timezones
# ------------------------------------------------------------

print("=== STEP 2: SESSION TIMEZONE ===")

session_timezone = spark.conf.get("spark.sql.session.timeZone")

print(f"Spark session timezone: {session_timezone}")

print()

# ------------------------------------------------------------
# Step 3 — Convert STRING to DateType Safely
#
# Our source has two possible date formats:
#
# yyyy-MM-dd
# dd/MM/yyyy
#
# Instead of assuming every record follows one format,
# try_to_timestamp() safely attempts each format.
#
# Invalid parsing returns NULL rather than failing.
#
# coalesce() then selects the first successfully parsed value.
#
# Finally, cast("date") converts the timestamp result to DateType.
# ------------------------------------------------------------

print("=== STEP 3: STRING → DATE ===")

date_format_1 = F.try_to_timestamp(
    F.col("order_date_str"),
    F.lit("yyyy-MM-dd"))

date_format_2 = F.try_to_timestamp(
    F.col("order_date_str"),
    F.lit("dd/MM/yyyy"))

parsed_order_date = F.coalesce(
    date_format_1,
    date_format_2
).cast("date")

df_converted = df_bronze.withColumn("order_date", parsed_order_date)

print("\nParsed dates:")

df_converted.select(
    "order_id",
    "order_date_str",
    "order_date"
).show(truncate=False)

print()



In [0]:
# ------------------------------------------------------------
# Step 4 — Convert STRING to TimestampType
#
# Source timestamp format:
#
# yyyy-MM-dd HH:mm:ss
#
# try_to_timestamp() is used so malformed source values
# can safely become NULL for later validation.
# ------------------------------------------------------------

print("=== STEP 4: STRING → TIMESTAMP ===")

df_converted = df_converted.withColumn(

    "order_timestamp",

    F.try_to_timestamp(
        F.col("order_ts_str"),
        F.lit("yyyy-MM-dd HH:mm:ss")))

df_converted.select(
    "order_id",
    "order_ts_str",
    "order_timestamp"
).show(truncate=False)

print()


In [0]:
# ------------------------------------------------------------
# Step 5 — Unix Epoch Seconds → Timestamp
#
# timestamp_seconds():
#
# Converts Unix epoch seconds into a Spark TimestampType.
#
# This is different from from_unixtime().
#
# from_unixtime():
#     returns a formatted STRING.
#
# timestamp_seconds():
#     returns TimestampType.
# ------------------------------------------------------------

print("=== STEP 5: UNIX EPOCH → TIMESTAMP ===")

df_converted = df_converted.withColumn(

    "unix_to_timestamp",

    F.timestamp_seconds(F.col("unix_ts")))

df_converted.select(
    "order_id",
    "unix_ts",
    "unix_to_timestamp"
).show(truncate=False)

print("""
Important:

from_unixtime(unix_ts)
→ formatted STRING

timestamp_seconds(unix_ts)
→ TimestampType

Timestamp display can depend on the Spark session timezone.
""")


In [0]:
# ------------------------------------------------------------
# Step 6 — Confirm Final Converted Schema
# ------------------------------------------------------------

print("=== STEP 6: CONVERTED SCHEMA ===")

df_converted.printSchema()

df_converted.select(
    "order_id",
    "order_date_str",
    "order_date",
    "order_ts_str",
    "order_timestamp",
    "unix_ts",
    "unix_to_timestamp"
).show(truncate=False)

print()

# ------------------------------------------------------------
# Step 7 — Extract Date Parts
#
# Common calendar extraction functions:
#
# year()
# month()
# dayofmonth()
# quarter()
# dayofweek()
# date_format()
#
# Spark dayofweek():
#
# 1 = Sunday
# 2 = Monday
# ...
# 7 = Saturday
# ------------------------------------------------------------

print("=== STEP 7: DATE PARTS ===")


df_parts = df_converted.select(

    F.col("order_id"),

    F.col("order_date"),

    F.year("order_date").alias("order_year"),

    F.month("order_date").alias("order_month"),

    F.dayofmonth("order_date").alias("order_day"),

    F.quarter("order_date").alias("order_quarter"),

    F.dayofweek("order_date").alias("day_of_week"),

    F.date_format("order_date","MMMM").alias("month_name"),

    F.date_format("order_date","EEEE").alias("day_name"))

df_parts.show(truncate=False)

print()


In [0]:
# ------------------------------------------------------------
# Step 8 — Extract Timestamp Parts
#
# Timestamp values contain both date and time information.
# ------------------------------------------------------------

print("=== STEP 8: TIMESTAMP PARTS ===")

df_time_parts = df_converted.select(

    "order_id",

    "order_timestamp",

    F.hour("order_timestamp").alias("hour"),

    F.minute("order_timestamp").alias("minute"),

    F.second("order_timestamp").alias("second"))

df_time_parts.show(truncate=False)

print()

# ------------------------------------------------------------
# Step 9 — Date Arithmetic
#
# date_add()
#     → Add days.
#
# date_sub()
#     → Subtract days.
#
# datediff()
#     → Difference in days.
#
# months_between()
#     → Approximate/fractional number of months between dates.
#
# current_date()
#     → Current Spark session date.
# ------------------------------------------------------------

print("=== STEP 9: DATE ARITHMETIC ===")


df_arith = df_converted.select(

    F.col("order_id"),

    F.col("order_date"),

    F.current_date().alias("run_date"),

    F.date_add(F.col("order_date"), 30).alias("due_date_30_days"),

    F.date_sub(F.col("order_date"), 7).alias("week_before"),

    F.datediff(F.current_date(),F.col("order_date")).alias("days_since_order"),

    F.round(F.months_between(F.current_date(),F.col("order_date")),2).alias("months_since_order"))

df_arith.show(truncate=False)

print("""
Important:

datediff()
→ Difference in DAYS.

months_between()
→ Can return a fractional month value.

Do not blindly cast months_between() to integer,
because doing so discards the fractional portion.

Also:

current_date()

changes according to the date on which the notebook runs.
""")


In [0]:
# ------------------------------------------------------------
# Step 10 — Weekend / Weekday Classification
#
# Spark dayofweek():
#
# Sunday   = 1
# Saturday = 7
#
# Therefore:
#
# isin(1, 7)
#
# identifies weekends.
# ------------------------------------------------------------

print("=== STEP 10: WEEKDAY / WEEKEND LOGIC ===")

df_week_type = df_converted.select(

    "order_id",
    "order_date",
    F.dayofweek("order_date").alias("day_of_week"),

    F.date_format("order_date","EEEE").alias("day_name"),

    F.when(F.dayofweek("order_date").isin(1,7),"Weekend")
    .otherwise("Weekday").alias("day_type"))

df_week_type.show(truncate=False)
print()


In [0]:
# ------------------------------------------------------------
# Step 11 — Parsing Validation
#
# The source contains 5 rows.
#
# One row has a different date format, but because we explicitly
# support both formats it should still parse successfully.
#
# The row containing source NULL remains NULL.
# ------------------------------------------------------------

print("=== STEP 11: PARSING VALIDATION ===")

non_null_source_dates = (df_bronze.filter(F.col("order_date_str").isNotNull()).count())

parsed_dates = (df_converted.filter(F.col("order_date").isNotNull()).count())

print(f"Non-NULL source date strings : " f"{non_null_source_dates}")

print(f"Successfully parsed dates    : " f"{parsed_dates}")

assert source_count == 5, ("Unexpected source row count")

assert non_null_source_dates == 4, ("Unexpected non-NULL source date count")

assert parsed_dates == 4, ("Expected four date values to parse")

print("Date parsing validation passed.")

print()

In [0]:
# ------------------------------------------------------------
# Step 12 — Real Use Case: samples.tpch.orders
#
# TPC-H order dates are already structured date values.
#
# Therefore:
#
# We do NOT need to parse them from STRING again.
#
# We can directly perform date analysis.
# ------------------------------------------------------------

print("=== STEP 12: REAL TPC-H ORDER DATE ANALYSIS ===")

df_orders = spark.table("samples.tpch.orders")

print("\nTPC-H orders schema:")

df_orders.select(
    "o_orderkey",
    "o_orderdate"
).printSchema()

df_date_analysis = df_orders.select(

    F.col("o_orderkey").alias("order_id"),

    F.col("o_orderdate").alias("order_date"),

    F.year("o_orderdate").alias("order_year"),

    F.month("o_orderdate").alias("order_month"),

    F.quarter("o_orderdate").alias("order_quarter"),

    F.dayofweek("o_orderdate").alias("day_of_week"),

    F.date_format("o_orderdate","EEEE").alias("day_name"),

    F.when(
        F.dayofweek(
            "o_orderdate").isin(1,7), "Weekend"
    ).otherwise("Weekday").alias("order_day_type"),

    F.datediff(
        F.current_date(),
        F.col("o_orderdate")).alias("days_since_order"))

print("\nOrders with date analysis:")

df_date_analysis.show(10, truncate=False)

print()


In [0]:
# ------------------------------------------------------------
# Step 13 — Weekend vs Weekday Aggregation
# ------------------------------------------------------------

print("=== STEP 13: WEEKEND vs WEEKDAY ORDERS ===")

df_weekend_summary = (

    df_date_analysis
    .groupBy("order_day_type")
    .agg(F.count("*").alias("order_count"))
    .orderBy("order_day_type"))

df_weekend_summary.show(truncate=False)

print()


In [0]:
# ------------------------------------------------------------
# Step 14 — Bronze → Silver Date Pattern
# ------------------------------------------------------------

print("=== STEP 14: BRONZE → SILVER DATE PATTERN ===")

print("""
Typical date-processing flow:

Bronze source
     ↓
Identify source date format
     ↓
Parse STRING safely
     ↓
DateType / TimestampType
     ↓
Validate failed parses
     ↓
Apply timezone rules where required
     ↓
Derive calendar attributes
     ↓
Apply business date logic
     ↓
Validate
     ↓
Silver-ready data


Important:

Do not treat:

DATE

TIMESTAMP

STRING date

Unix epoch

as interchangeable data types.

Each represents date/time information differently.
""")


In [0]:
# ------------------------------------------------------------
# Step 15 — Transformations vs Actions
# ------------------------------------------------------------

print(
    "=== STEP 15: TRANSFORMATIONS VS ACTIONS ==="
)


print("""
Date/time Column functions used:

try_to_timestamp()
timestamp_seconds()
year()
month()
dayofmonth()
quarter()
dayofweek()
date_format()
hour()
minute()
second()
date_add()
date_sub()
datediff()
months_between()
current_date()

These produce Spark Column expressions.

Transformations used:

select()
withColumn()
filter()
groupBy()
agg()
orderBy()


Actions used:

show()
count()

Spark still follows lazy evaluation.
""")


In [0]:
# ------------------------------------------------------------
# Step 16 — Final Learning Summary
# ------------------------------------------------------------

print(
    "=== FINAL LEARNING SUMMARY ==="
)


print("""
DateType:

Represents a calendar date.

Example:

2026-01-15


TimestampType:

Represents date + time.

Example:

2026-01-15 08:30:00


try_to_timestamp():

Safely parses STRING values into timestamps.

Malformed supported conversions can return NULL instead
of stopping the transformation.


STRING → DATE:

try_to_timestamp(...)
        ↓
cast("date")


Unix epoch seconds:

timestamp_seconds(unix_ts)
        ↓
TimestampType


Important:

from_unixtime()
→ returns STRING.

timestamp_seconds()
→ returns TimestampType.


Date extraction:

year()
month()
dayofmonth()
quarter()
dayofweek()
date_format()


Time extraction:

hour()
minute()
second()


Date arithmetic:

date_add()
date_sub()
datediff()
months_between()

dayofweek():

1 = Sunday
7 = Saturday

Timezone:

Timestamp interpretation/display can depend on the
Spark session timezone.

Silver-layer principle:

Raw date/time
     ↓
Understand source format
     ↓
Parse safely
     ↓
Apply correct data type
     ↓
Validate failed parsing
     ↓
Apply timezone/business rules
     ↓
Derive required attributes
     ↓
Silver-ready date/time
""")


print("=== S3.12 CELL 2 COMPLETED ===")

# S3.12 — Key Takeaways: Date and Time Functions

## 1. Core Date/Time Types

### DateType

Stores a calendar date only.

```text
2026-01-15
```

No time-of-day component.

---

### TimestampType

Stores date and time information.

```text
2026-01-15 08:30:00
```

Timestamp interpretation/display can be affected by the Spark
session timezone.

---

# 2. Conversion Functions

| Function | Typical Input | Output | Important Note |
|---|---|---|---|
| `to_date(col, format)` | String | DateType | Parse string according to expected date format |
| `to_timestamp(col, format)` | String | TimestampType | Parse string according to expected timestamp format |
| `try_to_timestamp(col, format)` | String | TimestampType | Invalid parse returns NULL |
| `from_unixtime(col)` | Unix seconds | String | Returns formatted string |
| `timestamp_seconds(col)` | Unix seconds | TimestampType | Converts epoch seconds directly to timestamp |

---

# 3. to_date()

Use when the source string represents a date.

Example:

```python
F.to_date(
    F.col("order_date_str"),
    "yyyy-MM-dd"
)
```

Example input:

```text
2026-01-15
```

Result:

```text
DateType
2026-01-15
```

The expected format should match the source-data contract.

---

# 4. to_timestamp()

Use when the source string contains date and time information.

Example:

```python
F.to_timestamp(
    F.col("order_ts_str"),
    "yyyy-MM-dd HH:mm:ss"
)
```

Example:

```text
2026-01-15 08:30:00
```

Result:

```text
TimestampType
```

---

# 5. try_to_timestamp() — Safe Parsing

`try_to_timestamp()` is useful when malformed input should become NULL
instead of stopping the transformation.

Example:

```python
F.try_to_timestamp(
    F.col("order_ts_str"),
    F.lit("yyyy-MM-dd HH:mm:ss")
)
```

Conceptually:

```text
"2026-01-15 08:30:00"
        ↓
valid parse
        ↓
Timestamp
```

while:

```text
"BAD_TIMESTAMP"
        ↓
try_to_timestamp()
        ↓
NULL
```

### Important

NULL after parsing still requires data-quality handling.

```text
Parse
   ↓
NULL
   ↓
Validate
   ↓
Keep / Reject / Quarantine
```

---

# 6. Multiple Incoming Date Formats

If a source legitimately supports more than one format, we can try
multiple parsing rules.

Example:

```python
F.coalesce(
    F.try_to_timestamp(
        F.col("date_str"),
        F.lit("yyyy-MM-dd")
    ),
    F.try_to_timestamp(
        F.col("date_str"),
        F.lit("dd/MM/yyyy")
    )
).cast("date")
```

Conceptually:

```text
Try format 1
     ↓
If NULL
     ↓
Try format 2
     ↓
First successful parse
```

Only support multiple formats when the source contract actually allows them.

---

# 7. Unix Epoch Conversion

Unix epoch values represent elapsed time from the Unix epoch.

For epoch seconds:

```python
F.timestamp_seconds(
    F.col("unix_ts")
)
```

returns:

```text
TimestampType
```

---

## from_unixtime()

```python
F.from_unixtime(
    F.col("unix_ts")
)
```

returns a formatted:

```text
StringType
```

Therefore:

```text
from_unixtime()
→ STRING

timestamp_seconds()
→ TIMESTAMP
```

Use the function that matches the required target data type.

---

# 8. Date Extraction Functions

Common functions:

```text
year()
month()
dayofmonth()
quarter()
dayofweek()
```

Example:

```python
F.year("order_date")
F.month("order_date")
F.dayofmonth("order_date")
F.quarter("order_date")
```

---

# 9. dayofweek()

Spark uses:

```text
1 = Sunday
2 = Monday
3 = Tuesday
4 = Wednesday
5 = Thursday
6 = Friday
7 = Saturday
```

Weekend example:

```python
F.dayofweek(
    "order_date"
).isin(
    1,
    7
)
```

---

# 10. Time Extraction Functions

For TimestampType values:

```text
hour()
minute()
second()
```

Example:

```python
F.hour("order_timestamp")
F.minute("order_timestamp")
F.second("order_timestamp")
```

---

# 11. date_format()

`date_format()` formats a date/timestamp as a string.

Example:

```python
F.date_format(
    "order_date",
    "MMMM"
)
```

Result:

```text
January
```

Example:

```python
F.date_format(
    "order_date",
    "EEEE"
)
```

Result:

```text
Monday
```

### Important

`date_format()` returns a STRING.

---

# 12. Date Arithmetic

## Add days

```python
F.date_add(
    "order_date",
    30
)
```

---

## Subtract days

```python
F.date_sub(
    "order_date",
    7
)
```

---

## Difference in days

```python
F.datediff(
    end_date,
    start_date
)
```

Conceptually:

```text
end_date - start_date
→ number of days
```

---

## Difference in months

```python
F.months_between(
    end_date,
    start_date
)
```

This can return a fractional value.

Example:

```text
8.93 months
```

Do not automatically cast the result to integer unless that matches
the business definition.

---

# 13. current_date()

```python
F.current_date()
```

returns the current Spark session date.

Useful for calculations such as:

```python
F.datediff(
    F.current_date(),
    F.col("order_date")
)
```

### Important

Results involving `current_date()` naturally change depending on
when the pipeline runs.

---

# 14. Timezone Considerations

Timezone matters primarily when processing timestamps.

Check the Spark session timezone with:

```python
spark.conf.get(
    "spark.sql.session.timeZone"
)
```

### Important

Do not assume:

```text
source timestamp
=
UTC timestamp
```

without understanding the source contract.

A common enterprise approach is to normalise cross-system event timestamps
to UTC while preserving source/local timezone information when the business
requires it.

The correct design depends on the system and business meaning.

---

# 15. Important Conversion Rules

### Rule 1

Do not leave date-like source strings as STRING if downstream processing
requires real date semantics.

Prefer:

```text
STRING
   ↓
Parse explicitly
   ↓
DateType / TimestampType
```

---

### Rule 2

Use `try_to_timestamp()` when malformed input should become NULL for
later validation instead of failing the transformation.

Do not treat it as a mandatory choice for every pipeline.

---

### Rule 3

Validate parsing results.

Example:

```text
Source non-NULL values
        ↓
Parsed non-NULL values
        ↓
Compare / investigate failures
```

---

### Rule 4

Understand timezone semantics before converting timestamps.

---

### Rule 5

Do not confuse:

```text
DateType
TimestampType
String
Unix epoch
```

They represent date/time information differently.

---

# 16. Extraction Quick Reference

| Requirement | Function |
|---|---|
| Year | `year()` |
| Month number | `month()` |
| Day of month | `dayofmonth()` |
| Quarter | `quarter()` |
| Day of week | `dayofweek()` |
| Month name | `date_format(..., "MMMM")` |
| Day name | `date_format(..., "EEEE")` |
| Hour | `hour()` |
| Minute | `minute()` |
| Second | `second()` |

---

# 17. Arithmetic Quick Reference

| Requirement | Function |
|---|---|
| Add days | `date_add()` |
| Subtract days | `date_sub()` |
| Difference in days | `datediff()` |
| Difference in months | `months_between()` |
| Current date | `current_date()` |

---

# 18. Silver-Layer Date Pattern

```text
Raw Date / Timestamp
        ↓
Understand source format
        ↓
Parse explicitly
        ↓
DateType / TimestampType
        ↓
Validate failed parses
        ↓
Apply timezone rules when required
        ↓
Extract required attributes
        ↓
Apply business date logic
        ↓
Validate
        ↓
Silver-ready date/time
```

---

# 19. Critical Rules

1. Parse date strings explicitly before relying on date semantics.

2. Use the format that matches the source contract.

3. `try_to_timestamp()` is useful when malformed parsing should return NULL.

4. `from_unixtime()` returns STRING.

5. `timestamp_seconds()` returns TimestampType.

6. `dayofweek()` uses Sunday = 1 and Saturday = 7.

7. `months_between()` can return fractional values.

8. `date_format()` returns STRING.

9. Timestamp interpretation/display can depend on session timezone.

10. Do not assume every timestamp should blindly be converted to UTC;
    follow the architecture and source/business timezone contract.

---

# 20. Easy Way to Remember

```text
STRING date?
→ Parse it

Need DateType?
→ to_date() / parsed timestamp cast to date

Need TimestampType?
→ to_timestamp() / try_to_timestamp()

Unix seconds?
→ timestamp_seconds()

Need year/month/day?
→ extraction functions

Need date difference?
→ datediff()

Need month difference?
→ months_between()

Need readable month/day name?
→ date_format()

Need timezone correctness?
→ check source contract + session timezone
```

---

# 21. Final S3.12 Understanding

```text
Raw date/time value
       ↓
Identify representation
       ↓
Parse correctly
       ↓
Validate
       ↓
Apply timezone rules
       ↓
Derive date/time attributes
       ↓
Apply business logic
       ↓
Silver-ready data
```

**S3.12 — Date and Time Functions completed.**